# Z3 & SMT Solving
### Binary Analysis Beyond Decompilers — CyberSphere Congress 2026

---

## What is Z3?

Z3 is an **SMT solver** from Microsoft Research.

You give it **variables** (unknowns) and **constraints** (rules), it gives you:
- `sat` → here is a solution
- `unsat` → no solution exists

> Instead of reversing what a binary computes — you describe what the input must satisfy, and Z3 finds it.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'z3-solver', '-q'])
from z3 import *
print('Z3 ready!')

---
## Example 1 Solving equations (the familiar case)

In [2]:
from z3 import *

x = Int('x')
y = Int('y')

s = Solver()
s.add(x + y == 10)
s.add(x - y == 4)

print(s.check())   # sat
print(s.model())   # [x = 7, y = 3]

sat
[y = 3, x = 7]


---
## Example 2 UNSAT: when no solution exists

In [4]:
from z3 import *

a = Int('a')
s = Solver()
s.add(a > 200)
s.add(a < 50)    # impossible

print(s.check())  # unsat
print("No value can be both > 200 and < 50")

unsat
No value can be both > 200 and < 50


---
## Example 3 BitVectors: how binaries actually work

Programs use fixed-width integers that wrap and use bitwise ops.  
`BitVec` models this exactly.

In [5]:
from z3 import *

# A binary checks: (x * 0x1337) & 0xffff == 0xdead
# What value of x passes this?

x = BitVec('x', 32)   # 32-bit unknown

s = Solver()
s.add((x * 0x1337) & 0xffff == 0xdead)

print(s.check())
if s.check() == sat:
    val = s.model()[x].as_long()
    print(f'x = {hex(val)}')
    print(f'Verify: {hex((val * 0x1337) & 0xffff)}')

sat
x = 0x973b
Verify: 0xdead


---
## Example 4 Multiple byte constraints (like a real crackme)

In [6]:
from z3 import *

# A binary checks 4 conditions on 4 input bytes [a, b, c, d]
a, b, c, d = [BitVec(n, 8) for n in 'abcd']

s = Solver()
s.add(a + b == 0x83)
s.add(b ^ c == 0x01)
s.add(c + d == 0x87)
s.add(d - a == 0x03)

# All bytes: printable ASCII (unsigned comparison!)
for v in [a, b, c, d]:
    s.add(UGE(v, 0x21))
    s.add(ULE(v, 0x7e))

print(s.check())
if s.check() == sat:
    m = s.model()
    result = bytes([m[v].as_long() for v in [a, b, c, d]])
    print(f'Bytes: {[hex(b) for b in result]}')
    print(f'ASCII: {result.decode()}')

sat
Bytes: ['0x5b', '0x28', '0x29', '0x5e']
ASCII: [()^


---
## Example 5 Find ALL solutions

In [7]:
from z3 import *

# How many 8-bit values satisfy x*x == 81 ?
x = BitVec('x', 8)
s = Solver()
s.add(x * x == 81)
s.add(x > 0)

solutions = []
while s.check() == sat:
    val = s.model()[x].as_long()
    solutions.append(val)
    s.add(x != val)   # exclude, find next

print(f'Solutions: {solutions}')
# 8-bit wrapping gives two answers: 9 and 247 (because 247 = -9 in signed 8-bit)

Solutions: [9, 119]


---
## Example 6 Prove MBA obfuscation 
The decompiler shows: `(x ^ y) + 2*(x & y)`  
You think it's just `x + y`. Prove it with Z3.

In [8]:
from z3 import *

x = BitVec('x', 32)
y = BitVec('y', 32)

s = Solver()
# Ask: is there ANY input where they differ?
s.add((x ^ y) + 2*(x & y) != x + y)

print(s.check())

if s.check() == unsat:
    print()
    print('PROVED: (x ^ y) + 2*(x & y) == x + y  for ALL 32-bit inputs')
    print()
    print('unsat = Z3 found NO counterexample.')
    print('They are always equal. This is how deobfuscation tools work.')

unsat

PROVED: (x ^ y) + 2*(x & y) == x + y  for ALL 32-bit inputs

unsat = Z3 found NO counterexample.
They are always equal. This is how deobfuscation tools work.


---

# Exercise Crack the Binary

You have a binary called `./crackme` that takes a 6-character password.

```bash
$ ./crackme hello
[-] Wrong password.

$ ./crackme ??????
[+] Correct! Flag: CTF{??????}
```

---

## What the decompiler shows you

Open `crackme` in Ghidra / IDA / [dogbolt.org](https://dogbolt.org).  
The `check()` function looks like this:

```c
int check(unsigned char *p) {
    if (strlen(p) != 6)              return 0;
    if ((p[0] + p[1]) != 0xb8)       return 0;
    if ((p[2] ^ p[3]) != 0x36)       return 0;
    if ((p[1] - p[2]) != 0x22)       return 0;
    if ((p[4] * 2) != (p[5] + 0x7f)) return 0;
    if ((p[3] + p[5]) != 0xda)       return 0;
    for (int i = 0; i < 6; i++)
        if (p[i] < 0x21 || p[i] > 0x7e) return 0;
    return 1;
}
```

Each `if (condition) return 0` is a **constraint**.  
Translate them all into Z3 and solve.

---

## Quick reference

```python
p = [BitVec(f'p{i}', 8) for i in range(6)]  # 6 unknown bytes

s.add(p[0] + p[1] == 0xb8)    # addition
s.add(p[2] ^ p[3] == 0x36)    # XOR
s.add(p[1] - p[2] == 0x22)    # subtraction
s.add(p[4] * 2 == p[5] + 0x7f) # multiplication

s.add(UGE(v, 0x21))  # unsigned >=
s.add(ULE(v, 0x7e))  # unsigned <=
```

---
## Your turn fill in the TODOs

In [13]:
from z3 import *

# 6 unknown bytes
p = [BitVec(f'p{i}', 8) for i in range(6)]

s = Solver()

# TODO 1: p[0] + p[1] == 0xb8

# TODO 2: p[2] ^ p[3] == 0x36

# TODO 3: p[1] - p[2] == 0x22

# TODO 4: p[4] * 2 == p[5] + 0x7f

# TODO 5: p[3] + p[5] == 0xda

# TODO 6: all bytes must be printable ASCII (0x21 to 0x7e)
for v in p:
    pass  # replace with UGE and ULE constraints

# TODO 7: check and print
print(s.check())

# TODO 8: extract the password
# if s.check() == sat:
#     m = s.model()
#     password = bytes([m[p[i]].as_long() for i in range(6)])
#     print(f'Password: {password.decode()}')

sat


---
## Complete Solution

In [16]:
from z3 import *
import subprocess

p = [BitVec(f'p{i}', 8) for i in range(6)]
s = Solver()

s.add(p[0] + p[1] == 0xb8)
s.add(p[2] ^ p[3] == 0x36)
s.add(p[1] - p[2] == 0x22)
s.add(p[4] * 2 == p[5] + 0x7f)
s.add(p[3] + p[5] == 0xda)

for v in p:
    s.add(UGE(v, 0x21))
    s.add(ULE(v, 0x7e))

print(s.check())

if s.check() == sat:
    m = s.model()
    password = bytes([m[p[i]].as_long() for i in range(6)])
    print(f'Password: {password.decode()}')

    result = subprocess.run(
        ['./crackme', password.decode()],
        capture_output=True, text=True
    )
    print(result.stdout.strip())

sat
Password: IoM{o_
[+] Correct! Flag: CTF{IoM{o_}


---
## Summary

| Concept | Code |
|---|---|
| Integer variable | `Int('x')` |
| 8-bit variable | `BitVec('x', 8)` |
| Add constraint | `s.add(condition)` |
| Solve | `s.check()` → `sat` / `unsat` |
| Get answer | `s.model()[x].as_long()` |
| Unsigned compare | `UGE(v, 0x21)` / `ULE(v, 0x7e)` |
| Find all solutions | loop + `s.add(x != found_value)` |